# PC-GNN Inspired Baseline

Implemented using GraphSAGE with dynamically sampled balanced batches and Focal Loss.

In [5]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, recall_score, roc_auc_score, precision_recall_curve, auc
from torch_geometric.data import Data
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
DATA_DIR = '../data/elliptic++/'

Using device: cuda


In [6]:
print("Loading data...")
wallet_features = pd.read_csv(DATA_DIR + 'wallets_features.csv').drop_duplicates(subset='address', keep='first')
wallet_classes = pd.read_csv(DATA_DIR + 'wallets_classes.csv')
edges = pd.read_csv(DATA_DIR + 'AddrAddr_edgelist.csv')

wallet_merged = wallet_features.merge(wallet_classes, on='address', how='left')

addr_to_idx = {addr: i for i, addr in enumerate(wallet_merged['address'].values)}

feature_cols = [c for c in wallet_merged.columns if c not in ['address', 'Time step', 'class']]
X = torch.tensor(wallet_merged[feature_cols].values, dtype=torch.float32)

class_vals = wallet_merged['class'].fillna(3).values
y_mapped = np.zeros_like(class_vals, dtype=np.int64) - 1
y_mapped[class_vals == 1] = 1 # Illicit
y_mapped[class_vals == 2] = 0 # Licit
Y = torch.tensor(y_mapped, dtype=torch.long)

edge_src = [addr_to_idx[src] for src in edges['input_address'] if src in addr_to_idx]
edge_dst = [addr_to_idx[dst] for dst in edges['output_address'] if dst in addr_to_idx]
edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

time_steps = wallet_merged['Time step'].values

train_mask = torch.tensor((time_steps <= 31) & (y_mapped != -1), dtype=torch.bool)
val_mask = torch.tensor(((time_steps >= 32) & (time_steps <= 34)) & (y_mapped != -1), dtype=torch.bool)
test_mask = torch.tensor((time_steps >= 35) & (y_mapped != -1), dtype=torch.bool)

data = Data(x=X, edge_index=edge_index, y=Y, train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)
data = T.ToUndirected()(data)
data = data.to(device)

print(data)


Loading data...
Data(x=[822942, 55], edge_index=[2, 5553655], y=[822942], train_mask=[822942], val_mask=[822942], test_mask=[822942])


In [7]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss)
        return focal_loss.mean()

class PC_GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index)
        return x

# 4. Training with dynamic manual sampling (mimicking Pick and Choose)
def train_full_batch_sampled(model, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    
    # Identify training nodes
    train_idx = data.train_mask.nonzero(as_tuple=False).view(-1)
    train_y = data.y[train_idx]
    
    # 50/50 Sampling (like ImbalancedSampler)
    illicit_idx = train_idx[train_y == 1]
    licit_idx = train_idx[train_y == 0]
    
    # Subsample majority class to match minority class size
    perm = torch.randperm(licit_idx.size(0))[:illicit_idx.size(0)]
    sampled_licit_idx = licit_idx[perm]
    
    # Combine for this epoch's loss calculation
    balanced_idx = torch.cat([illicit_idx, sampled_licit_idx])
    
    out_balanced = out[balanced_idx]
    y_balanced = data.y[balanced_idx]
    
    loss = criterion(out_balanced, y_balanced)
    loss.backward()
    optimizer.step()
    return loss.item()

def test_full_batch(model, mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        probs = F.softmax(out[mask], dim=1)
        preds = out[mask].argmax(dim=1).cpu().numpy()
        probs_illicit = probs[:, 1].cpu().numpy()
        y_true = data.y[mask].cpu().numpy()
        
        f1 = f1_score(y_true, preds, pos_label=1, zero_division=0)
        f1_micro = f1_score(y_true, preds, average='micro', zero_division=0)
        recall = recall_score(y_true, preds, pos_label=1, zero_division=0)
        try:
            auc_test = roc_auc_score(y_true, probs_illicit)
            precision, recall_curve_vals, _ = precision_recall_curve(y_true, probs_illicit)
            auprc = auc(recall_curve_vals, precision)
        except ValueError:
            auc_test, auprc = 0.0, 0.0
            
    return f1, f1_micro, recall, auc_test, auprc

In [8]:
model = PC_GraphSAGE(data.x.shape[1], 128, 2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = FocalLoss(alpha=0.75, gamma=2.0)

best_val_f1 = 0
best_test_metrics = None
patience, no_improve = 50, 0
epochs = 300

print("Training PC-GNN Inspired Strong Baseline (Full Batch with Focal Loss & Balanced Sampling)...")
for epoch in range(1, epochs + 1):
    loss = train_full_batch_sampled(model, optimizer, criterion)
    val_f1, val_f1_micro, val_rec, val_auc, val_auprc = test_full_batch(model, data.val_mask)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_test_metrics = test_full_batch(model, data.test_mask)
        no_improve = 0
    else:
        no_improve += 1
        
    if epoch % 20 == 0:
        print(f'Epoch {epoch:03d} | Loss: {loss:.4f} | Val F1: {val_f1:.4f} | Val Rec: {val_rec:.4f}')
        
    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

test_f1, test_f1_micro, test_rec, test_auc, test_auprc = best_test_metrics
print("="*50)
print("FINAL PC-GNN INSPIRED BASELINE RESULTS (Minority Class: Illicit)")
print("="*50)
print(f"GraphSAGE + FocalLoss + Random Balanced Under-sampling:")
print(f"F1 (Illicit): {test_f1:.4f}")
print(f"Micro-F1: {test_f1_micro:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"AUC-ROC  : {test_auc:.4f}")
print(f"AUPRC    : {test_auprc:.4f}")


Training PC-GNN Inspired Strong Baseline (Full Batch with Focal Loss & Balanced Sampling)...
Epoch 020 | Loss: 0.0694 | Val F1: 0.2189 | Val Rec: 1.0000
Epoch 040 | Loss: 0.0611 | Val F1: 0.3782 | Val Rec: 0.9319
Early stopping at epoch 51
FINAL PC-GNN INSPIRED BASELINE RESULTS (Minority Class: Illicit)
GraphSAGE + FocalLoss + Random Balanced Under-sampling:
F1 (Illicit): 0.2686
Micro-F1: 0.8240
Recall   : 0.6112
AUC-ROC  : 0.7938
AUPRC    : 0.1520
